In [ ]:
# 1. Install dependencies
!pip install onnxruntime onnxmltools skl2onnx xgboost

In [ ]:
# Cell 2: Mount Google Drive and load data
from google.colab import drive
import pandas as pd
import json

drive.mount('/content/drive')

# --- Update these paths if your folder structure is different ---
CSV_PATH = '/content/drive/MyDrive/FLSS_ML/scheduling_dataset.csv'
JSON_PATH = '/content/drive/MyDrive/FLSS_ML/encoders.json'
# ---------------------------------------------------------------

df = pd.read_csv(CSV_PATH)

# Load encoders to get the exact column schema and day encoding
with open(JSON_PATH, 'r') as f:
    encoders = json.load(f)

schema = encoders['schema']
day_encoding = encoders['day_encoding']

print(f"✅ Loaded {len(df)} rows")
print(f"📋 Schema: {schema}")
print(f"📅 Day encoding: {day_encoding}")

In [ ]:
# Cell 3: Prepare features (X) and target (y)

# Target: the continuous match score (0.0 to 1.0)
y = df['match_score']

# Features: all columns EXCEPT the target, in the same order as the schema
# This ordering is critical — Angular must send features in this exact order
features_list = [col for col in schema if col != 'match_score']
X = df[features_list]

print(f"✅ Features ({len(features_list)} columns): {features_list}")
print(f"✅ Target distribution:")
print(y.describe())


In [ ]:
# Cell 4: Temporal split by semester
train_mask = df['academic_year_id'] == 3
test_mask  = df['academic_year_id'] == 5

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f"🎓 Training rows  : {len(X_train)}")
print(f"🧪 Test rows      : {len(X_test)}")


In [ ]:
# Cell 5: Train XGBoost Regressor
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    objective='reg:squarederror',  # Continuous output (0.0–1.0)
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=20           # Print progress every 20 rounds
)

print("✅ Training complete!")


In [ ]:
# Cell 6: Evaluate model quality
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

preds = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
r2   = r2_score(y_test, preds)

print("=" * 40)
print(f"  RMSE (lower is better)  : {rmse:.4f}")
print(f"  R² Score (1.0 = perfect): {r2:.4f}")
print("=" * 40)

# Flag prediction range issues
clipped = ((preds < 0) | (preds > 1)).sum()
print(f"  Out-of-range predictions: {clipped} rows "
      f"(Angular will clip these to 0–1)")


In [ ]:
# Cell 7: Feature importance — understand which features the model uses
import matplotlib.pyplot as plt

importances = model.feature_importances_
feat_df = pd.DataFrame({
    'Feature': features_list,
    'Importance': importances
}).sort_values('Importance', ascending=True)

feat_df.plot(
    kind='barh', x='Feature', y='Importance',
    title='Feature Importance', figsize=(8, 5), legend=False
)
plt.tight_layout()
plt.show()


In [ ]:
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType

# Rename model's internal feature names to f0, f1, ...
# This aligns with onnxmltools's expectation for XGBoost feature names.
# It's important that the order of these renamed features matches the
# order of features in `features_list` and during model training.
feature_names_for_onnx = [f'f{i}' for i in range(len(features_list))]
model.get_booster().feature_names = feature_names_for_onnx

# Number of input features must match Angular's Float32Array size
n_features = len(features_list)
initial_type = [('float_input', FloatTensorType([None, n_features]))]

onnx_model = onnxmltools.convert_xgboost(
    model, initial_types=initial_type
)

# Define the path where the ONNX model will be saved
ONNX_SAVE_PATH = '/content/drive/MyDrive/FLSS_ML/model.onnx' # You can change this path

# Save to the specified path
with open(ONNX_SAVE_PATH, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"✅ model.onnx saved to {ONNX_SAVE_PATH}! (Input shape: [batch, {n_features}])")
print(f"📋 Feature order Angular must follow: {features_list}")